## Hypothesis Testing

In this section, I test whether weather conditions and time-related factors have a statistically significant impact on food delivery demand.

These tests are based on the patterns observed during the exploratory data analysis. In particular, the EDA suggested that time-related factors may be more influential than weather variables.

In [7]:
import pandas as pd
from scipy.stats import pearsonr, ttest_ind

## Data Preparation

Before running the hypothesis tests, the datasets need to be prepared in the same way as in the EDA section.

The order dataset is converted into hourly order counts, and the weather dataset is aligned to the same hourly datetime structure. After that, both datasets are merged into a single dataframe.

In [8]:
# Load datasets
df_orders = pd.read_csv("order_history_kaggle_data.csv")
df_weather = pd.read_excel("weather.xlsx")

# Convert order timestamps into hourly datetime
df_orders["datetime"] = pd.to_datetime(
    df_orders["Order Placed At"],
    format="%I:%M %p, %B %d %Y",
    errors="coerce"
).dt.floor("h")

df_orders = df_orders.dropna(subset=["datetime"]).copy()

# Convert weather datetime into hourly format
df_weather["datetime"] = pd.to_datetime(
    df_weather["datetime"],
    errors="coerce"
).dt.floor("h")

df_weather = df_weather.dropna(subset=["datetime"]).copy()

# Create hourly order counts
orders_per_hour = df_orders.groupby("datetime").size().reset_index(name="order_count")

# Merge datasets
df_merged = pd.merge(orders_per_hour, df_weather, on="datetime", how="inner")

# Extra variables for time-based tests
df_merged["hour"] = df_merged["datetime"].dt.hour
df_merged["is_weekend"] = df_merged["datetime"].dt.dayofweek >= 5

print("Merged dataset shape:", df_merged.shape)
print("Date range:", df_merged["datetime"].min(), "→", df_merged["datetime"].max())
df_merged.head()

Merged dataset shape: (2555, 7)
Date range: 2024-09-01 00:00:00 → 2025-01-31 23:00:00


,datetime,order_count,time,temperature_2m,precipitation,hour,is_weekend
0,2024-09-01 00:00:00,1,2024-09-01T00:00,25.9,0.0,0,True
1,2024-09-01 01:00:00,2,2024-09-01T01:00,27.0,0.0,1,True
2,2024-09-01 02:00:00,7,2024-09-01T02:00,28.1,0.0,2,True
3,2024-09-01 03:00:00,5,2024-09-01T03:00,29.5,0.0,3,True
4,2024-09-01 11:00:00,3,2024-09-01T11:00,33.2,0.0,11,True


## Testing Functions

To keep the analysis organized and consistent, I define reusable functions for correlation tests and group comparison tests.

The correlation test is used for continuous variables such as temperature and precipitation, while the group comparison test is used for comparisons such as weekday versus weekend and peak versus non-peak hours.

In [9]:
def correlation_hypothesis_test(x_var, y_var, df):
    subset = df[[x_var, y_var]].dropna()
    x = subset[x_var]
    y = subset[y_var]

    corr, p_val = pearsonr(x, y)

    return {
        "sample_size": len(subset),
        "correlation": corr,
        "p_value": p_val
    }

def group_difference_test(group1, group2):
    group1 = group1.dropna()
    group2 = group2.dropna()

    t_stat, p_val = ttest_ind(group1, group2, equal_var=False)

    return {
        "group1_mean": group1.mean(),
        "group2_mean": group2.mean(),
        "t_statistic": t_stat,
        "p_value": p_val
    }

## Test 1: Temperature vs Order Demand

This test examines whether temperature is associated with changes in food delivery demand.

Pearson correlation is used to measure the strength of the relationship between temperature and hourly order count.

In [10]:
temp_results = correlation_hypothesis_test("temperature_2m", "order_count", df_merged)

print("Sample size:", temp_results["sample_size"])
print("Correlation:", temp_results["correlation"])
print("P-value:", temp_results["p_value"])

Sample size: 2555
Correlation: -0.0698112675043187
P-value: 0.00041347123127316983



The results show that the relationship between temperature and order demand is weak.

The p-value suggests that temperature does not have a statistically significant effect on demand.

## Test 2: Precipitation vs Order Demand

This test evaluates whether rainfall has an impact on food delivery demand.

Pearson correlation is used to identify any potential relationship.

In [11]:
prec_results = correlation_hypothesis_test("precipitation", "order_count", df_merged)

print("Sample size:", prec_results["sample_size"])
print("Correlation:", prec_results["correlation"])
print("P-value:", prec_results["p_value"])

Sample size: 2555
Correlation: -0.025730717319171797
P-value: 0.19353600365684384



The results indicate that precipitation does not show a strong relationship with order demand.

There is no clear statistical evidence that rainfall significantly affects customer behavior.

## Test 3: Weekday vs Weekend Demand

This test compares food delivery demand between weekdays and weekends.

A t-test is used to determine whether there is a meaningful difference between the two groups.

In [12]:
weekend_orders = df_merged[df_merged["is_weekend"]]["order_count"]
weekday_orders = df_merged[~df_merged["is_weekend"]]["order_count"]

week_results = group_difference_test(weekend_orders, weekday_orders)

print("Weekend mean:", week_results["group1_mean"])
print("Weekday mean:", week_results["group2_mean"])
print("T-statistic:", week_results["t_statistic"])
print("P-value:", week_results["p_value"])

Weekend mean: 9.556944444444444
Weekday mean: 7.869209809264305
T-statistic: 5.270897391849206
P-value: 1.6095303460396182e-07



This test evaluates whether customer behavior changes across the week.

A low p-value would indicate that demand differs between weekdays and weekends.

## Test 4: Peak vs Non-Peak Hours

This test evaluates whether food delivery demand is higher during peak meal hours.

Peak hours are defined as typical lunch and dinner periods.

In [13]:
peak_orders = df_merged[
    ((df_merged["hour"] >= 11) & (df_merged["hour"] <= 14)) |
    ((df_merged["hour"] >= 18) & (df_merged["hour"] <= 21))
]["order_count"]

non_peak_orders = df_merged[
    ~(((df_merged["hour"] >= 11) & (df_merged["hour"] <= 14)) |
      ((df_merged["hour"] >= 18) & (df_merged["hour"] <= 21)))
]["order_count"]

peak_results = group_difference_test(peak_orders, non_peak_orders)

print("Peak mean:", peak_results["group1_mean"])
print("Non-peak mean:", peak_results["group2_mean"])
print("T-statistic:", peak_results["t_statistic"])
print("P-value:", peak_results["p_value"])

Peak mean: 10.53923205342237
Non-peak mean: 6.407516580692705
T-statistic: 15.144915302989233
P-value: 6.119646879194682e-49



The results show that demand is higher during peak hours compared to non-peak hours.

This aligns with the patterns observed in the EDA, where demand increased during lunch and dinner times.

This suggests that time of day is one of the strongest drivers of food delivery demand.

## Overall Conclusion

The hypothesis tests confirm that time-related factors have a stronger influence on food delivery demand compared to weather variables.

While temperature and precipitation show weak relationships, demand clearly increases during specific hours of the day.

This indicates that customer behavior is primarily driven by daily routines rather than environmental conditions.